# Stage 1 — Data Ingestion
### Credit Card Fraud Detection · Tier A pipeline

**Purpose.** Acquire the raw transaction file into the analysis environment, prove it is the file we
think it is, and record its lineage — *before* a single analytical decision is made.

This notebook does four things and nothing else:

1. **Load** with an explicit dtype contract (no silent type coercion).
2. **Hash and log lineage** — source path, SHA-256, size, timestamp, shape.
3. **Validate the schema** against the data dictionary in `DOCS/creditcard_dataset_analysis.md`, one
   rule at a time, pass/fail.
4. **Record the governance position** — what is and isn't personally identifying here, and what that
   forecloses downstream.

> **Immutability decision.** The source CSV is 144 MB. Rather than duplicate it into `data/raw/`
> (a folder convention), immutability is enforced by **never writing to the source path** and by
> recording its **SHA-256**, which every downstream notebook re-verifies. The guarantee is the hash,
> not the copy.

**Governing plan:** `IMPLEMENTATION_PLAN.md` §2 · **Standard:** `DOCS/STRUCTURE.md` Stage 1

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Resolve the project root whether this runs from notebooks/ or the project root.
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

RAW_SRC = PROJECT_ROOT / "creditcard.csv" / "creditcard.csv"
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
for d in (DATA_RAW, DATA_PROCESSED):
    d.mkdir(parents=True, exist_ok=True)

assert RAW_SRC.exists(), f"raw source not found: {RAW_SRC}"

print(f"project root : {PROJECT_ROOT.name}")
print(f"raw source   : {RAW_SRC.relative_to(PROJECT_ROOT)}")
print(f"size on disk : {RAW_SRC.stat().st_size / 1024**2:,.1f} MB")
print(f"python       : {platform.python_version()}  |  pandas {pd.__version__}  |  numpy {np.__version__}")

project root : Credit Card Fraud Detection
raw source   : creditcard.csv\creditcard.csv
size on disk : 143.8 MB
python       : 3.14.6  |  pandas 3.0.3  |  numpy 2.5.1


## 1.1 Load under an explicit dtype contract

The header quotes every field name **and** quotes the `Class` values (`"0"` / `"1"`). Left to
inference that column could arrive as `object`. We declare the contract up front so a violation is a
loud failure rather than a silent one that surfaces three notebooks later.

In [2]:
V_COLS = [f"V{i}" for i in range(1, 29)]
DTYPE_CONTRACT = {"Time": "float64", "Amount": "float64", "Class": "int8"}
DTYPE_CONTRACT.update({c: "float64" for c in V_COLS})

t0 = time.perf_counter()
df = pd.read_csv(RAW_SRC, dtype=DTYPE_CONTRACT)
load_s = time.perf_counter() - t0

print(f"loaded in {load_s:,.1f}s  ->  {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\ndtypes honoured: {(df.dtypes.astype(str).to_dict() == {**DTYPE_CONTRACT})}")
print(f"Class dtype    : {df['Class'].dtype}  (quoted in the CSV — confirmed parsed as integer)")
df.head(3)

loaded in 2.3s  ->  284,807 rows x 31 columns

dtypes honoured: True
Class dtype    : int8  (quoted in the CSV — confirmed parsed as integer)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0


In [3]:
mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"in-memory footprint : {mem_mb:,.1f} MB  (vs {RAW_SRC.stat().st_size / 1024**2:,.1f} MB on disk)")
print(f"per-row             : {df.memory_usage(deep=True).sum() / len(df):,.0f} bytes")
print("\nThe working set is comfortable in float64. Downcasting to float32 is deferred to the")
print("modelling notebook (05), where it buys fit speed; the stored processed table stays float64.")

in-memory footprint : 65.5 MB  (vs 143.8 MB on disk)
per-row             : 241 bytes

The working set is comfortable in float64. Downcasting to float32 is deferred to the
modelling notebook (05), where it buys fit speed; the stored processed table stays float64.


## 1.2 Lineage — source, hash, timestamp, shape

Recorded to `data/raw/_lineage.csv`. The SHA-256 is the contract every downstream notebook checks
against: if it changes, the analysis is invalid until re-run.

In [4]:
def sha256_of(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

t0 = time.perf_counter()
FILE_HASH = sha256_of(RAW_SRC)
print(f"SHA-256 computed in {time.perf_counter() - t0:,.1f}s")

lineage = pd.DataFrame([{
    "artifact": "creditcard.csv",
    "source_path": str(RAW_SRC.relative_to(PROJECT_ROOT)).replace("\\", "/"),
    "provenance": "ULB Machine Learning Group / Kaggle — European cardholders, September 2013",
    "sha256": FILE_HASH,
    "size_bytes": RAW_SRC.stat().st_size,
    "pull_timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "n_rows": df.shape[0],
    "n_cols": df.shape[1],
    "load_seconds": round(load_s, 2),
    "stage": "1 — ingestion",
    "immutability": "source never written to; hash re-verified downstream (no 144MB copy)",
}])
lineage.to_csv(DATA_RAW / "_lineage.csv", index=False)

print(f"\nSHA-256 : {FILE_HASH}")
lineage.T

SHA-256 computed in 0.2s

SHA-256 : 76274b691b16a6c49d3f159c883398e03ccd6d1ee12d9d8ee38f4b4b98551a89


,0
artifact,creditcard.csv
source_path,creditcard.csv/creditcard.csv
provenance,ULB Machine Learning Group / Kaggle — European...
sha256,76274b691b16a6c49d3f159c883398e03ccd6d1ee12d9d...
size_bytes,150828752
pull_timestamp_utc,2026-07-22T07:41:16+00:00
n_rows,284807
n_cols,31
load_seconds,2.3
stage,1 — ingestion


## 1.3 Schema validation — one rule per row, pass/fail

Every rule below comes from the data dictionary in `DOCS/creditcard_dataset_analysis.md`. Three of
them are load-bearing rather than cosmetic:

- **`Class` split is exactly 284,315 / 492.** If it isn't, this is not the canonical ULB file and
  every published comparison in the report would be invalid.
- **`Time` is monotonically non-decreasing.** The temporal Day 0 → Day 1 split planned in notebook 05
  depends on the file being in transaction order. If it isn't sorted, that split is meaningless.
- **`V1`–`V28` each have mean ≈ 0.** These are PCA components; a non-zero mean would mean the file has
  been transformed or subset since publication. This is a genuine integrity check, not decoration.

In [5]:
EXPECTED_COLS = ["Time"] + V_COLS + ["Amount", "Class"]
class_counts = df["Class"].value_counts()
v_means = df[V_COLS].mean()
v_stds = df[V_COLS].std()

rules = [
    ("shape",        "row count is exactly 284,807",          df.shape[0] == 284_807,                      f"{df.shape[0]:,}"),
    ("shape",        "column count is exactly 31",             df.shape[1] == 31,                           f"{df.shape[1]}"),
    ("schema",       "column names and order match dictionary", list(df.columns) == EXPECTED_COLS,          "exact match" if list(df.columns) == EXPECTED_COLS else "MISMATCH"),
    ("completeness", "zero nulls across all 31 columns",       int(df.isna().sum().sum()) == 0,             f"{int(df.isna().sum().sum()):,} nulls"),
    ("target",       "Class takes only values {0, 1}",         set(df["Class"].unique()) == {0, 1},         str(sorted(df['Class'].unique()))),
    ("target",       "legitimate count is exactly 284,315",    int(class_counts.get(0, 0)) == 284_315,      f"{int(class_counts.get(0, 0)):,}"),
    ("target",       "fraud count is exactly 492",             int(class_counts.get(1, 0)) == 492,          f"{int(class_counts.get(1, 0)):,}"),
    ("range",        "Amount >= 0 everywhere",                 bool((df["Amount"] >= 0).all()),             f"min {df['Amount'].min():,.2f}"),
    ("range",        "Amount max ~ 25,691.16",                 abs(df["Amount"].max() - 25_691.16) < 0.01,  f"max {df['Amount'].max():,.2f}"),
    ("range",        "Time within [0, 172792]",                bool(df["Time"].between(0, 172_792).all()),  f"[{df['Time'].min():,.0f}, {df['Time'].max():,.0f}]"),
    ("ordering",     "Time is monotonically non-decreasing",   bool(df["Time"].is_monotonic_increasing),    "sorted" if df["Time"].is_monotonic_increasing else "UNSORTED"),
    ("integrity",    "V1-V28 all finite (no inf/NaN)",         bool(np.isfinite(df[V_COLS].to_numpy()).all()), "all finite"),
    ("integrity",    "V1-V28 means all ~ 0 (PCA property)",    bool((v_means.abs() < 1e-8).all()),          f"max |mean| = {v_means.abs().max():.2e}"),
    ("integrity",    "V1-V28 std dev strictly positive",       bool((v_stds > 0).all()),                    f"min std = {v_stds.min():.3f}"),
]

schema = pd.DataFrame(rules, columns=["category", "rule", "passed", "observed"])
schema.insert(0, "rule_id", [f"R{i:02d}" for i in range(1, len(schema) + 1)])
schema["result"] = np.where(schema["passed"], "PASS", "FAIL")
schema.drop(columns="passed").to_csv(DATA_RAW / "_schema_validation.csv", index=False)

n_fail = int((schema["result"] == "FAIL").sum())
print(f"{len(schema) - n_fail}/{len(schema)} rules PASS,  {n_fail} FAIL\n")
schema.drop(columns="passed")

14/14 rules PASS,  0 FAIL



,rule_id,category,rule,observed,result
0,R01,shape,"row count is exactly 284,807","284,807",PASS
1,R02,shape,column count is exactly 31,31,PASS
2,R03,schema,column names and order match dictionary,exact match,PASS
3,R04,completeness,zero nulls across all 31 columns,0 nulls,PASS
4,R05,target,"Class takes only values {0, 1}","[np.int8(0), np.int8(1)]",PASS
5,R06,target,"legitimate count is exactly 284,315","284,315",PASS
6,R07,target,fraud count is exactly 492,492,PASS
7,R08,range,Amount >= 0 everywhere,min 0.00,PASS
8,R09,range,"Amount max ~ 25,691.16","max 25,691.16",PASS
9,R10,range,"Time within [0, 172792]","[0, 172,792]",PASS


In [6]:
if n_fail:
    failed = schema.loc[schema["result"] == "FAIL", "rule"].tolist()
    raise AssertionError(
        "Schema validation failed — this is NOT the canonical ULB creditcard.csv. "
        "Do not proceed to cleaning until resolved. Failing rules: " + "; ".join(failed)
    )
print("Schema contract satisfied. This is the canonical ULB file; downstream comparisons are valid.")

Schema contract satisfied. This is the canonical ULB file; downstream comparisons are valid.


## 1.4 The number that governs the whole project

Class balance is not one finding among many here — it dictates the metric, the split, the model
weighting, and the report's opening line. It is confirmed at ingestion so nothing downstream can
quietly assume otherwise.

In [7]:
n_total = len(df)
n_fraud = int(class_counts[1])
n_legit = int(class_counts[0])
fraud_rate = n_fraud / n_total
imbalance_ratio = n_legit / n_fraud

print(f"total transactions : {n_total:,}")
print(f"legitimate         : {n_legit:,}  ({n_legit / n_total:.4%})")
print(f"fraud              : {n_fraud:,}  ({fraud_rate:.4%})")
print(f"imbalance ratio    : {imbalance_ratio:,.0f} : 1   ->  roughly 1 fraud in every {imbalance_ratio:,.0f} transactions")
print()
print(f"Consequence, stated now so it cannot be forgotten later:")
print(f"  a model that predicts 'never fraud' scores {1 - fraud_rate:.2%} accuracy and catches 0 of {n_fraud} frauds.")
print(f"  Accuracy is therefore retired as a metric from this point forward; AUPRC leads.")
print(f"  A random ranker's average precision baseline is {fraud_rate:.6f} — that is the bar to beat.")

total transactions : 284,807
legitimate         : 284,315  (99.8273%)
fraud              : 492  (0.1727%)
imbalance ratio    : 578 : 1   ->  roughly 1 fraud in every 578 transactions

Consequence, stated now so it cannot be forgotten later:
  a model that predicts 'never fraud' scores 99.83% accuracy and catches 0 of 492 frauds.
  Accuracy is therefore retired as a metric from this point forward; AUPRC leads.
  A random ranker's average precision baseline is 0.001727 — that is the bar to beat.


In [8]:
span_seconds = df["Time"].max() - df["Time"].min()
print(f"observation window : {span_seconds:,.0f} seconds = {span_seconds / 3600:,.1f} hours = {span_seconds / 86400:.2f} days")
print(f"transactions/hour  : {n_total / (span_seconds / 3600):,.0f} average")
print()
print("Two days is the hard ceiling on this project: no seasonality, no month-end effects, no concept")
print("drift measurement. Recorded here; it resurfaces as a first-class limitation in the report.")

observation window : 172,792 seconds = 48.0 hours = 2.00 days
transactions/hour  : 5,934 average

Two days is the hard ceiling on this project: no seasonality, no month-end effects, no concept
drift measurement. Recorded here; it resurfaces as a first-class limitation in the report.


## 1.5 Governance — what this data is, and what its anonymity forecloses

The defining property of this dataset is that **it arrives already pseudonymized**: V1–V28 are
principal components, and the original transaction attributes (merchant, geography, device,
card-present flag) were destroyed in the transformation.

That has a benign consequence and a costly one, and both are recorded here rather than discovered later:

- **Benign:** there is no PII to protect. No cardholder ID, no merchant ID, no geography, no
  demographics. Re-identification risk is low.
- **Costly:** there are **no demographic attributes**, which means a protected-subgroup fairness audit
  on this model is **not merely skipped — it is impossible**. `DOCS/STRUCTURE.md` makes a fairness
  audit mandatory for any model affecting people, and a fraud scorer plainly does. The impossibility
  is logged here and is promoted to a main-narrative limitation in the final report, where it caps the
  deployment recommendation.
- Also costly: **no cardholder key** means per-card velocity features — the strongest signals in real
  fraud detection — cannot be built at all.

In [9]:
governance = pd.DataFrame([
    {"item": "PII present",                "status": "None",        "detail": "No cardholder, merchant, device, or geographic identifier in any column."},
    {"item": "Pseudonymization",           "status": "Upstream",    "detail": "V1-V28 are PCA components; original attributes destroyed by the publisher before release."},
    {"item": "Re-identification risk",     "status": "Low",         "detail": "No quasi-identifier set. Amount + Time alone cannot isolate an individual without external linkage."},
    {"item": "Demographic attributes",     "status": "ABSENT",      "detail": "BLOCKS the mandatory protected-subgroup fairness audit. Escalated to a report limitation, not omitted."},
    {"item": "Entity key (cardholder)",    "status": "ABSENT",      "detail": "Per-card velocity / deviation-from-own-average features are unbuildable. Caps achievable performance."},
    {"item": "Label provenance",           "status": "UNKNOWN",     "detail": "Unclear whether Class is confirmed (post-investigation) or flagged fraud. Ground-truth trust is bounded."},
    {"item": "Purpose limitation",         "status": "Compliant",   "detail": "Public research dataset released for fraud-detection research; this is that purpose."},
    {"item": "Retention / distribution",   "status": "Local only",  "detail": "The 144MB CSV is never committed; .gitignore ships notebooks, report HTML and README only."},
])
governance.to_csv(DATA_RAW / "_governance.csv", index=False)
governance

,item,status,detail
0,PII present,None,"No cardholder, merchant, device, or geographic..."
1,Pseudonymization,Upstream,V1-V28 are PCA components; original attributes...
2,Re-identification risk,Low,No quasi-identifier set. Amount + Time alone c...
3,Demographic attributes,ABSENT,BLOCKS the mandatory protected-subgroup fairne...
4,Entity key (cardholder),ABSENT,Per-card velocity / deviation-from-own-average...
5,Label provenance,UNKNOWN,Unclear whether Class is confirmed (post-inves...
6,Purpose limitation,Compliant,Public research dataset released for fraud-det...
7,Retention / distribution,Local only,The 144MB CSV is never committed; .gitignore s...


## 1.6 First look — the shape of what we're working with

No analysis here; a sanity read of the two columns that survived anonymization with their meaning
intact. Everything else is a principal component and will be treated as such.

In [10]:
summary = df[["Time", "Amount"] + V_COLS[:4]].describe().T
summary["skew"] = df[["Time", "Amount"] + V_COLS[:4]].skew()
print("Time and Amount are the ONLY business-interpretable columns in the file.\n")
summary.round(3)

Time and Amount are the ONLY business-interpretable columns in the file.



,count,mean,std,min,25%,50%,75%,max,skew
Time,284807.0,94813.86,47488.146,0.000,54201.500,84692.000,139320.500,172792.000,-0.036
Amount,284807.0,88.35,250.120,0.000,5.600,22.000,77.165,25691.160,16.978
V1,284807.0,0.00,1.959,-56.408,-0.920,0.018,1.316,2.455,-3.281
V2,284807.0,0.00,1.651,-72.716,-0.599,0.065,0.804,22.058,-4.625
V3,284807.0,-0.00,1.516,-48.326,-0.890,0.180,1.027,9.383,-2.240
V4,284807.0,0.00,1.416,-5.683,-0.849,-0.020,0.743,16.875,0.676


In [11]:
print("Amount, per class — pooled statistics would be 99.83% legitimate traffic and hide the subject:")
print()
amt = df.groupby("Class")["Amount"].agg(n="size", mean="mean", median="median",
                                        q25=lambda s: s.quantile(0.25),
                                        q75=lambda s: s.quantile(0.75), max="max")
amt.index = ["legitimate", "fraud"]
display(amt.round(2))
print("\nA first, unverified signal: the fraud median sits below the legitimate median despite fraud")
print("carrying a heavier mean. Tested properly in notebook 04 — flagged here, not concluded.")

Amount, per class — pooled statistics would be 99.83% legitimate traffic and hide the subject:



,n,mean,median,q25,q75,max
legitimate,284315,88.29,22.00,5.65,77.05,25691.16
fraud,492,122.21,9.25,1.00,105.89,2125.87



A first, unverified signal: the fraud median sits below the legitimate median despite fraud
carrying a heavier mean. Tested properly in notebook 04 — flagged here, not concluded.


In [12]:
INGESTION_FACTS = {
    "source_sha256": FILE_HASH,
    "n_rows_raw": int(n_total),
    "n_cols_raw": int(df.shape[1]),
    "n_fraud": int(n_fraud),
    "n_legit": int(n_legit),
    "fraud_rate": float(fraud_rate),
    "imbalance_ratio": float(imbalance_ratio),
    "always_legit_accuracy": float(1 - fraud_rate),
    "window_seconds": float(span_seconds),
    "window_hours": float(span_seconds / 3600),
    "amount_max": float(df["Amount"].max()),
    "schema_rules_total": int(len(schema)),
    "schema_rules_passed": int(len(schema) - n_fail),
}
(PROJECT_ROOT / "reports").mkdir(exist_ok=True)
with open(PROJECT_ROOT / "reports" / "_key_figures.json", "w") as fh:
    json.dump({"stage_1_ingestion": INGESTION_FACTS}, fh, indent=2)

print("reports/_key_figures.json seeded with Stage 1 facts.")
print("Every number the final report quotes flows through this file — the report hardcodes nothing.")
print(json.dumps(INGESTION_FACTS, indent=2))

reports/_key_figures.json seeded with Stage 1 facts.
Every number the final report quotes flows through this file — the report hardcodes nothing.
{
  "source_sha256": "76274b691b16a6c49d3f159c883398e03ccd6d1ee12d9d8ee38f4b4b98551a89",
  "n_rows_raw": 284807,
  "n_cols_raw": 31,
  "n_fraud": 492,
  "n_legit": 284315,
  "fraud_rate": 0.001727485630620034,
  "imbalance_ratio": 577.8760162601626,
  "always_legit_accuracy": 0.9982725143693799,
  "window_seconds": 172792.0,
  "window_hours": 47.99777777777778,
  "amount_max": 25691.16,
  "schema_rules_total": 14,
  "schema_rules_passed": 14
}


---

## Stage 1 — Gate Checklist (`DOCS/STRUCTURE.md`)

- [x] **Raw data secured, untouched** — source never written to; SHA-256 recorded and re-verified downstream, in place of a 144 MB duplicate copy (decision documented above)
- [x] **Source, timestamp, and row/column counts logged** — `data/raw/_lineage.csv`
- [x] **Schema matches expectations** — 14/14 rules PASS, including the canonical 284,315 / 492 class split, `Time` monotonicity, and the PCA mean-zero integrity check
- [x] **PII / data governance review completed** — `data/raw/_governance.csv`; the *absence* of demographics is logged as the blocker for the mandatory fairness audit, and the absence of a cardholder key as the ceiling on feature engineering

### What carries forward

| Fact established | Where it binds |
|---|---|
| 0.172% fraud rate, 578:1 imbalance | Metric choice (AUPRC), class weighting, stratified splits — notebooks 03–05 |
| `Time` is sorted, spans exactly 2 days | Makes the Day 0 → Day 1 temporal split viable — notebook 05 |
| No demographic attributes | Fairness audit is impossible; escalated to a report limitation — notebooks 05–06 |
| No cardholder key | Per-card velocity features unbuildable; caps achievable performance — notebook 02 |
| Label provenance unknown | Bounds trust in ground truth; motivates the unsupervised cross-check — notebook 05 |

**Next:** `02_cleaning.ipynb` — verify the reported 1,081 exact duplicates, decide their fate on
leakage grounds, and derive the row-local time/amount columns.